# StrataForge Real PDF Parser + Tree Demo Notebook
## Phase 01 + Phase 02 Manual Demo
Purpose: exercise the current parser substrate and deterministic tree pipeline against the local real document `903000608.pdf`.


### Environment Assumptions
- This notebook is a local operator demo and is not CI acceptance evidence.
- It requires `903000608.pdf` at the repository root.
- It prefers existing local parse artifacts under `notebooks/artifacts/parse_runs/` when they match the real PDF fingerprint.
- If a fresh parse is required, local Tesseract and tessdata must be installed for OCR-marked pages.


In [ ]:
# environment setup
from pathlib import Path

REPO_ROOT = Path.cwd()
PDF_PATH = REPO_ROOT / "903000608.pdf"
LOCAL_ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "artifacts" / "parse_runs"
EXECUTION_ARTIFACT_ROOT = REPO_ROOT / "notebooks" / "_artifacts" / "spec_v1_parser_tree_demo"
LOCAL_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
EXECUTION_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(
        "Real demo notebook requires 903000608.pdf at the repository root."
    )

print(f"repo_root={REPO_ROOT}")
print(f"pdf_path={PDF_PATH}")
print(f"local_artifact_root={LOCAL_ARTIFACT_ROOT}")
print(f"execution_artifact_root={EXECUTION_ARTIFACT_ROOT}")


In [ ]:
# imports
import json
import shutil
from typing import Any

from strataforge.domain import ParseRequest, ParserSettings, TreeBuildRequest
from strataforge.ingest import parse_document
from strataforge.ingest.fingerprint import fingerprint_document
from strataforge.tree import TreeConflictError, build_tree


In [ ]:
# configuration
PARSE_RUN_ID = "spec_v1_run"
TREE_RUN_ID = "spec_v1_tree"
REPRESENTATIVE_PAGE_INDEXES = (11, 23, 32, 34, 35)
TESSDATA_CANDIDATES = (
    Path("/usr/share/tesseract-ocr/5/tessdata"),
    Path("/usr/share/tesseract-ocr/4.00/tessdata"),
    Path("/usr/share/tesseract-ocr/tessdata"),
)
LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS = {
    "ledger_path": "ledger/page-ledger.jsonl",
    "selected_outline_path": "outline/selected.json",
    "source_copy_path": "source/original.pdf",
    "pymupdf_outline_path": "outline/pymupdf.normalized.json",
    "pymupdf_rich_outline_path": "outline/pymupdf.rich.json",
    "pypdf_outline_path": "outline/pypdf.normalized.json",
}


def load_json(path: Path) -> Any:
    return json.loads(path.read_text(encoding="utf-8"))


def load_json_lines(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]


def resolve_artifact_path(parse_root: Path, stored_path: str | None) -> Path | None:
    if stored_path is None:
        return None
    path = Path(stored_path)
    return path if path.is_absolute() else parse_root / path


def resolve_tessdata_path() -> str | None:
    for candidate in TESSDATA_CANDIDATES:
        if candidate.is_dir():
            return str(candidate)
    return None


def manifest_artifacts_are_usable(manifest_path: Path, payload: dict[str, Any]) -> bool:
    parse_root = manifest_path.parent
    for key in LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS:
        resolved = resolve_artifact_path(parse_root, payload.get(key))
        if resolved is None or not resolved.exists():
            return False
    return True


def expected_page_relative_path(page_index: int, leaf_name: str) -> str:
    return f"pages/{page_index:06d}/{leaf_name}"


def normalize_local_ledger_rows(ledger_path: Path) -> list[dict[str, Any]]:
    rows = load_json_lines(ledger_path)
    parse_root = ledger_path.parent.parent
    normalized_rows: list[dict[str, Any]] = []
    changed = False
    for row in rows:
        page_index = row["page_index"]
        normalized = dict(row)
        expected_paths = {
            "native_text_artifact_path": expected_page_relative_path(page_index, "native.txt"),
            "native_rawdict_artifact_path": expected_page_relative_path(page_index, "native.rawdict.json.gz"),
            "ocr_text_artifact_path": expected_page_relative_path(page_index, "ocr.txt"),
            "ocr_rawdict_artifact_path": expected_page_relative_path(page_index, "ocr.rawdict.json.gz"),
            "render_artifact_path": expected_page_relative_path(page_index, "render.png"),
            "ocr_render_artifact_path": expected_page_relative_path(page_index, "render.png"),
        }
        for key, relative_path in expected_paths.items():
            current_path = resolve_artifact_path(parse_root, normalized.get(key))
            expected_path = parse_root / relative_path
            if current_path is not None and current_path.exists():
                continue
            if expected_path.exists():
                normalized[key] = relative_path
                changed = True
            elif key in {
                "native_rawdict_artifact_path",
                "ocr_text_artifact_path",
                "ocr_rawdict_artifact_path",
                "render_artifact_path",
                "ocr_render_artifact_path",
            }:
                if normalized.get(key) is not None:
                    changed = True
                normalized[key] = None

        text_artifact_expected = (
            normalized.get("ocr_text_artifact_path")
            if normalized.get("ocr_text_artifact_path") is not None
            else normalized["native_text_artifact_path"]
        )
        if normalized.get("text_artifact_path") != text_artifact_expected:
            normalized["text_artifact_path"] = text_artifact_expected
            changed = True
        normalized_rows.append(normalized)

    if changed:
        ledger_path.write_text(
            "\n".join(json.dumps(row, sort_keys=True, ensure_ascii=True) for row in normalized_rows)
            + "\n",
            encoding="utf-8",
        )
    return normalized_rows


def normalize_local_manifest_payload(
    manifest_path: Path,
    payload: dict[str, Any],
) -> dict[str, Any]:
    parse_root = manifest_path.parent
    normalized = dict(payload)
    normalized["artifact_root"] = str(parse_root)
    for key, relative_path in LOCAL_MANIFEST_EXPECTED_RELATIVE_PATHS.items():
        expected = parse_root / relative_path
        if expected.exists():
            normalized[key] = relative_path
    return normalized


def resolve_existing_parse_manifest(document_id: str, sha256: str) -> tuple[Path | None, str]:
    candidate = LOCAL_ARTIFACT_ROOT / PARSE_RUN_ID / document_id / "manifest.json"
    if not candidate.exists():
        return None, "missing_local_manifest"
    payload = load_json(candidate)
    if payload.get("fingerprint", {}).get("sha256") != sha256:
        return None, "mismatched_local_manifest"
    if manifest_artifacts_are_usable(candidate, payload):
        return candidate, "reused_local_manifest"

    normalized = normalize_local_manifest_payload(candidate, payload)
    if manifest_artifacts_are_usable(candidate, normalized):
        candidate.write_text(
            json.dumps(normalized, indent=2, sort_keys=True, ensure_ascii=True),
            encoding="utf-8",
        )
        return candidate, "reused_local_manifest"
    return None, "stale_local_manifest"


def ensure_parse_manifest() -> tuple[dict[str, Any], Path, str]:
    pdf_fingerprint = fingerprint_document(str(PDF_PATH))
    manifest_path, mode = resolve_existing_parse_manifest(
        pdf_fingerprint.document_id,
        pdf_fingerprint.sha256,
    )
    if manifest_path is not None:
        return load_json(manifest_path), manifest_path, mode

    parse_run_root = LOCAL_ARTIFACT_ROOT / PARSE_RUN_ID
    if parse_run_root.exists():
        shutil.rmtree(parse_run_root)

    tessdata_path = resolve_tessdata_path()
    if tessdata_path is None:
        raise RuntimeError(
            f"No reusable local parse manifest was found ({mode}) and no tessdata path was detected for a fresh parse."
        )

    manifest = parse_document(
        ParseRequest(
            source_path=str(PDF_PATH),
            parse_run_id=PARSE_RUN_ID,
            artifact_root=str(LOCAL_ARTIFACT_ROOT),
            settings=ParserSettings(tessdata_path=tessdata_path),
        )
    )
    manifest_path = Path(manifest.artifact_root) / "manifest.json"
    return load_json(manifest_path), manifest_path, "parsed_fresh"


In [ ]:
# execution
parse_manifest, parse_manifest_path, parse_mode = ensure_parse_manifest()
parse_root = parse_manifest_path.parent
selected_outline_path = resolve_artifact_path(parse_root, parse_manifest["selected_outline_path"])
ledger_path = resolve_artifact_path(parse_root, parse_manifest["ledger_path"])
if selected_outline_path is None or ledger_path is None:
    raise RuntimeError("Resolved parse manifest is missing required artifact paths.")

selected_outline = load_json(selected_outline_path)
ledger_rows = normalize_local_ledger_rows(ledger_path)
pdf_fingerprint = parse_manifest["fingerprint"]

tree_manifest = None
for candidate_tree_run_id in (
    TREE_RUN_ID,
    f"{TREE_RUN_ID}-strategy-v1",
    f"{TREE_RUN_ID}-{pdf_fingerprint['sha256'][:8]}-major-changes",
):
    try:
        tree_manifest = build_tree(
            TreeBuildRequest(
                parse_manifest_path=str(parse_manifest_path),
                tree_run_id=candidate_tree_run_id,
            )
        )
        break
    except TreeConflictError:
        continue
if tree_manifest is None:
    raise RuntimeError("Unable to resolve a reusable local tree_run_id for the demo notebook.")
tree_manifest_path = Path(tree_manifest.artifact_root) / "manifest.json"
build_report = load_json(Path(tree_manifest.build_report_path))
verification_report = load_json(Path(tree_manifest.verification_report_path))
node_cards = load_json(Path(tree_manifest.node_cards_path))
committed_nodes = load_json(Path(tree_manifest.committed_hierarchy_path))
unassigned_spans = load_json(Path(tree_manifest.unassigned_spans_path))


In [ ]:
# execution
ledger_by_page = {row["page_index"]: row for row in ledger_rows}
representative_pages = []
for page_index in REPRESENTATIVE_PAGE_INDEXES:
    row = ledger_by_page.get(page_index)
    if row is None:
        continue
    native_path = resolve_artifact_path(parse_root, row["native_text_artifact_path"])
    native_excerpt = None
    if native_path is not None and native_path.exists():
        native_excerpt = native_path.read_text(encoding="utf-8")[:200]
    representative_pages.append(
        {
            "page_index": page_index,
            "page_label": row["page_label"],
            "extraction_method": row["extraction_method"],
            "needs_ocr": row["needs_ocr"],
            "native_text_artifact_path": row["native_text_artifact_path"],
            "text_length": row["text_length"],
            "native_excerpt": native_excerpt,
        }
    )

top_level_node_cards = [card for card in node_cards if card["level"] == 1][:10]
failed_node_results = [
    {
        "subject_id": result["subject_id"],
        "status": result["status"],
        "issue_codes": [issue["code"] for issue in result.get("issues", [])],
    }
    for result in verification_report["node_results"]
    if result["status"] != "passed"
][:5]


In [ ]:
# inspect results
summary = {
    "pdf": {
        "path": str(PDF_PATH),
        "sha256": pdf_fingerprint["sha256"],
        "page_count": pdf_fingerprint["page_count"],
        "parse_mode": parse_mode,
    },
    "parse": {
        "manifest_path": str(parse_manifest_path),
        "outline_source": parse_manifest["selected_outline_source"],
        "page_count": parse_manifest["page_count"],
        "ocr_page_count": sum(1 for row in ledger_rows if row["needs_ocr"]),
        "selected_outline_entries": len(selected_outline["entries"]),
    },
    "outline_preview": selected_outline["entries"][:6],
    "representative_pages": representative_pages,
    "tree": {
        "manifest_path": str(tree_manifest_path),
        "committed_node_count": tree_manifest.committed_node_count,
        "unassigned_span_count": tree_manifest.unassigned_span_count,
        "verification_status": verification_report["status"],
        "outline_trust_mode": build_report["outline_trust_mode"],
    },
    "top_level_node_cards": [
        {
            "title": card["title"],
            "level": card["level"],
            "page_span": card["page_span"],
        }
        for card in top_level_node_cards
    ],
    "failed_node_results": failed_node_results,
    "unassigned_spans_preview": unassigned_spans[:5],
}
print(json.dumps(summary, indent=2, sort_keys=True))


### Known Limitations
- This notebook depends on a local real PDF and local notebook artifact state, so it is not part of mandatory CI acceptance.
- If the matching parse manifest is missing, a fresh parse may require local Tesseract and tessdata for OCR-marked pages.
- The tree output reflects the current deterministic Phase 01 + Phase 02 pipeline only; it is not a Phase 04 multimodal preview.
